# T-GNN: Temporal Heterogeneous GNN for Credential Fraud Detection

**Google Colab notebook** — imports all modules from the repository.
No code is duplicated here; everything runs through the project modules.

> 🔒 **Data privacy**: This notebook uses only synthetic data. No real Corilla customer data is loaded.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────
!pip install torch-geometric pyarrow pyyaml tqdm -q

In [ ]:
# ── 2. Clone / mount repo ─────────────────────────────────────
# Option A: clone from your Git remote
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# %cd YOUR_REPO

# Option B: mount Google Drive where repo is stored
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/TGNN-code

import os, sys
# Uncomment and set the path after cloning/mounting:
# os.chdir('/content/TGNN-code')
# sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

In [ ]:
# ── 3. Generate synthetic dataset ────────────────────────────
from data.generate_synthetic import generate
import yaml
with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

SEED = 42
generate(SEED, cfg, 'data/synthetic/')
print('Dataset generated.')

In [ ]:
# ── 4. Validate dataset ───────────────────────────────────────
!python data/validate_dataset.py --data data/synthetic/

In [ ]:
# ── 5. Load dataset ───────────────────────────────────────────
from data.temporal_dataset import TemporalCredentialDataset
dataset    = TemporalCredentialDataset('data/synthetic/', n_snapshots=60)
snapshots  = list(dataset)
input_dims = dataset.get_input_dims()
print('Input dims:', input_dims)
print('Example snapshot:', snapshots[0])

In [ ]:
# ── 6. Build T-GNN model ─────────────────────────────────────
import torch
from models.temporal_gnn import TemporalHeteroGNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = TemporalHeteroGNN(
    input_dims=input_dims,
    embedding_dim=128, gru_hidden=128,
    n_layers=2, dropout=0.3,
).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── 7. Train (quick 5-epoch demo) ────────────────────────────
from graph.snapshots import chronological_split
from training.train import train_one_epoch, compute_class_weight
from training.evaluate import evaluate

train_ids, val_ids, test_ids = chronological_split(60)
train_s = [snapshots[i] for i in train_ids]
val_s   = [snapshots[i] for i in val_ids]
test_s  = [snapshots[i] for i in test_ids]

pos_w = compute_class_weight(train_s, cfg)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(1, 6):
    loss = train_one_epoch(model, train_s, opt, device, pos_w)
    vm   = evaluate(model, val_s, device)
    print(f'Epoch {epoch}  loss={loss:.4f}  val_f1={vm["f1"]:.4f}  val_auc={vm["roc_auc"]:.4f}')

In [ ]:
# ── 8. Full training (100 epochs, all seeds) ─────────────────
# Uncomment to run. This is equivalent to:
# !python experiments/run_baselines.py --seeds 42 123 2024 3407 7777
# !python experiments/run_ablation.py --seeds 42 123 2024 3407 7777

print('Uncomment the lines above to run full experiments.')

In [ ]:
# ── 9. Generate figures (after experiments) ───────────────────
# !python figures/figure5.py
# !python figures/figure6.py
# !python figures/figure7.py
print('Run figures after completing experiments.')